In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import sklearn.base
import sklearn.compose
import sklearn.inspection
import sklearn.linear_model
import sklearn.model_selection
import sklearn.preprocessing
from IPython.core.display import display

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s ames

# Linear regression

## Univariate Linear regression


### Introduction

Linear regression is very interesting to code yourself: it is not too complex but allows you to touch on many machine learning concepts.

We will continue our work on the house price data.

Let's spend a few seconds in python to load the dataset:

In [ ]:
train_df = pd.read_csv("ames/train.csv", index_col="Id")

In [ ]:
display(train_df)

### Columns Used

In this lab, we'll only use one feature, `GrLivArea`, and `SalePrice` as the target.

- *Plot `GrLivArea` against `SalePrice`* using [`sns.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html?highlight=jointplot#seaborn.jointplot)
- *Define then normalize `x` and `y`*. For the normalization, you can:
  - either use [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
  - or compute the normalization yourself $$\mathbf{x}_{norm}=\frac{\mathbf{x} - \bar{\mathbf{x}}}{\sigma_\mathbf{x}}$$

In [ ]:
# seaborn.jointplot(???)
# plt.show()

In [ ]:
# Normalize X
# Your code here

# Normalize Y
# Your code here

#### Solution

In [ ]:
sns.jointplot(x="GrLivArea", y="SalePrice", data=train_df)
plt.show()

In [ ]:
# Standardize x
x_scaler = sklearn.preprocessing.StandardScaler()
x = x_scaler.fit_transform(train_df[["GrLivArea"]])[:, 0]

# Standardize y
y_scaler = sklearn.preprocessing.StandardScaler()
y = y_scaler.fit_transform(train_df[["SalePrice"]])[:, 0]

### Linear Regression Example
To understand the joint evolution of a house's sale price and living area, we'll restrict ourselves to a strong hypothesis: we can find an affine function that correctly models the relationship between inputs and outputs. As a reminder, an affine function has the form $f(x) = ax + b$.

Geometrically, this function can be represented by a line.

For instance, here, we'd like to find the following hypothesis:

In [ ]:
sns.regplot(x=x, y=y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

All this is promising, but here we have used [`seaborn`](https://seaborn.pydata.org/) to calculate the parameters of the hypothesis that best represents the data. We will now see how to do this ourselves.

### Estimating the Parameters

To estimate $a$ and $b$, we need 2 things:

- a cost function, which tells us, for given parameters $a$ and $b$, whether we're doing well or not
- a method to estimate the best parameters given this cost function

### Predictions

As a prerequisite to computing the cost function, we need to know how to compute predictions.

To apply the linear transformation defined by $a$ and $b$ to $\mathbf{x}$, we simply use the multiplication and addition of scalars to a vector: $a\mathbf{x} + b$

*Define the `predict` function so that it computes the linear transformation given by $a$ & $b$.*

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
  pass  # Your code here

#### Solution

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
  return a * x + b


x_example = np.array([1, 2, 3])
y_example = np.array([4, 10, 3])
a_example = 3
b_example = 2

print(predict(x_example, a_example, b_example))

### Cost Function

The most widely used cost function is the mean of the squared differences between the actual points and those obtained by the hypothesis with the current parameters.

Denoting $E$ the cost function and $r$ the number of training examples, we can write:

$$
  E(a, b) = \sum\frac{(a\mathbf{x} + b - \mathbf{y})^2}{2r}
$$

- *Define the `residuals` function using the `predict` function*

- *Define the `cost` function using the `residuals` function*

In [ ]:
def residuals(
  x: np.ndarray,
  y: np.ndarray,
  a: float,
  b: float,
) -> np.ndarray:
  pass  # Your code here


def cost(
  x: np.ndarray,
  y: np.ndarray,
  a: float,
  b: float,
) -> np.float32:
  pass  # Your code here

#### Solution

In [ ]:
def residuals(
  x: np.ndarray,
  y: np.ndarray,
  a: float,
  b: float,
) -> np.ndarray:
  return predict(x, a, b) - y


print("Residuals:", residuals(x_example, y_example, a_example, b_example))


def cost(
  x: np.ndarray,
  y: np.ndarray,
  a: float,
  b: float,
) -> np.float32:
  return np.mean(residuals(x, y, a, b) ** 2) / 2


print("Costs:", cost(x_example, y_example, a_example, b_example))

### Optimizing the Parameters

Optimizing the parameters means minimizing the cost function:

$$\min_{a, b} \sum\frac{(a\mathbf{x} + b - \mathbf{y})^2}{2r}$$

There's a closed-form formula for this, which we won't use in this lab for two reasons: it doesn't apply to very large datasets, and the method we'll use can be reused for logistic regression and other techniques (general neural networks, gradient boosted trees, etc).

We'll use gradient descent. This iterative method takes successive steps following the derivative (to maximize) or its opposite (to minimize) of the loss function. In the case of linear regression, it converges to the global optimum (the best solution).

In pseudo-code, the algorithm is (with the right gradients computed):

$$
\begin{aligned}
& \text{while not converged:} \\
& \quad a \leftarrow a - \alpha \frac{\mathbf{x}^T(a\mathbf{x} + b - \mathbf{y})}{r} \\
& \quad b \leftarrow b - \alpha \sum\frac{a\mathbf{x_i} + b - \mathbf{y_i}}{r} \\
\end{aligned}
$$

where $\alpha$ is the learning rate.


*Implement the `gradient` function using the `residuals` function.*

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  pass  # Your code here

#### Solution

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  r = residuals(x, y, a, b)
  g_a = (x @ r) / y.size
  g_b = r.mean()
  return g_a, g_b

### Gradient descent

*Implement gradient descent in a function `gradient_descent`.*

In [ ]:
def gradient_descent(
  x: np.ndarray, y: np.ndarray, alpha: float, nb_iter: int, epsilon: float
) -> tuple[float, float, list[float]]:
  pass  # Your code here

#### Solution

In [ ]:
def gradient_descent(
  x: np.ndarray,
  y: np.ndarray,
  alpha: float = 0.1,
  nb_iter: int = 1000,
  epsilon: float = 10e-6,
) -> tuple[float, float, list[float]]:
  a = random.random()
  b = random.random()
  print(f"Initial (random) parameters: a = {a:.2f}, b = {b:.2f}")
  initial_cost = cost(x, y, a, b)
  print(f"Initial cost: {initial_cost:.2f}")
  costs = [initial_cost]
  for i in range(nb_iter):
    g_a, g_b = gradient(x, y, a, b)
    a -= alpha * g_a
    b -= alpha * g_b
    costs.append(cost(x, y, a, b))
    if costs[-2] - costs[-1] < epsilon:
      print(f"Stopping: not enough progress at iteration {i}")
      break
  print(f"Final parameters: a = {a:.2f}, b = {b:.2f}")
  print(f"Final cost: {costs[-1]:.2f}")
  return a, b, costs

To finish, the call to the gradient_descent function

In [ ]:
a, b, costs = gradient_descent(x, y, 0.1, 1000)
print(f"Successive costs:  {', '.join(map(str, costs))}")

### Checking Our Hypothesis

Let's now check the trained model with a few plots:

- predicted values against actual values
- residuals
- training costs

In [ ]:
inv_x = x_scaler.inverse_transform(x[:, None])[:, 0]
inv_y = y_scaler.inverse_transform(y[:, None])[:, 0]
inv_yh = y_scaler.inverse_transform(predict(x, a, b)[:, None])[:, 0]

xs = np.arange(0, 6_000, 100)[:, None]
ys = y_scaler.inverse_transform(predict(x_scaler.transform(xs)[:, 0], a, b)[:, None])
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("Predictions and actual values")
plt.xlabel("Living area")
plt.ylabel("Sale price")
plt.show()

plt.plot(inv_x, inv_yh - inv_y, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Living area")
plt.ylabel("Residuals")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Training costs during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Error")
plt.show()

### Correcting the Positive Skew

In [ ]:
sns.distplot(train_df[["SalePrice"]], fit=scipy.stats.norm)
plt.title("SalePrice distribution before normalization")
plt.show()

# Standardize Y
Y_scaler_log = sklearn.preprocessing.StandardScaler()
Y = Y_scaler_log.fit_transform(np.log1p(train_df[["SalePrice"]]))[:, 0]

sns.distplot(Y, fit=scipy.stats.norm)
plt.title("SalePrice distribution after normalization")
plt.show()

In [ ]:
a, b, costs = gradient_descent(x, Y, 0.01, 5000)

In [ ]:
inv_X = x_scaler.inverse_transform(x[:, None])[:, 0]
inv_Y = np.expm1(Y_scaler_log.inverse_transform(y[:, None])[:, 0])
inv_Yh = np.expm1(Y_scaler_log.inverse_transform(predict(x, a, b)[:, None])[:, 0])

Xs = np.arange(0, 6_000, 100).reshape(-1, 1)
Ys = np.expm1(Y_scaler_log.inverse_transform(predict(x_scaler.transform(Xs), a, b)))

In [ ]:
plt.plot(inv_X, inv_Y, "b.", alpha=0.10)
plt.plot(Xs, Ys, "r")
plt.title("Predictions and actual values")
plt.xlabel("Living area")
plt.ylabel("Sale price")
plt.show()

plt.plot(inv_X, inv_Yh - inv_Y, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Living area")
plt.ylabel("Residuals")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Training costs during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Error")
plt.show()

## Multivariate Linear Regression

To continue this practical work, we will repeat a full pre-processing run on the entire AMES data (not just the `GrLiveArea` column).

In [ ]:
!pip install nevergrad
import nevergrad

In [ ]:
def preprocess(train_file: str, test_file: str) -> np.ndarray:
  train_X = pd.read_csv(train_file, index_col="Id")
  test_X = pd.read_csv(test_file, index_col="Id")

  train_Y = train_X[["SalePrice"]]
  train_X = train_X.drop(columns=["SalePrice"])

  all_X = pd.concat([train_X, test_X])

  # Fill with median
  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  # Fill with mode
  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  # Fill with 0
  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  # Other fills
  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  # Give all other NAs the string value NA, which will be a category
  all_X = all_X.fillna("NA")

  # Turn the numerical codes into strings so that they're treated as
  # a categorical variable
  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Number of NAs: {all_X.isnull().sum().sum()}")

  dummies = pd.get_dummies(all_X)
  return (
    dummies.iloc[: train_X.shape[0], :].values,
    train_Y.values,
    dummies.iloc[train_X.shape[0] :, :].values,
    dummies.columns,
  )

In [ ]:
X_train, Y_train, X_test, columns = preprocess(
  "ames/train.csv", "ames/test.csv"
)

### Normalizing the Output Variable

Use [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to center and scale the targets into the new variable `Y_train_scaled`.

In [ ]:
# your code here

#### Solution

In [ ]:
Y_scaler = sklearn.preprocessing.StandardScaler()
Y_train_scaled = Y_scaler.fit_transform(Y_train)

### Simple linear model training

- Use [`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) to train a linear regression on normalized data.
- Display the coefficients of the learned regression

In [ ]:
# your code here

#### Solution

In [ ]:
linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(X_train, Y_train_scaled)
print(linear_regression.coef_)

In [ ]:
Y_scaler.inverse_transform(linear_regression.predict(X_train[:2]))

### Cross validation

Use cross validation with [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) on a linear regression model and normalized data.

In [ ]:
# your code here

#### Solution

In [ ]:
def score(
  model: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train_scaled,
):
  scores = sklearn.model_selection.cross_val_score(model, X, Y, cv=5, scoring="r2")
  return sum(scores) / len(scores)


def score(
  model: sklearn.base.BaseEstimator, X: np.ndarray = X_train, Y: np.ndarray = Y_train
) -> float:
  pipeline = sklearn.compose.TransformedTargetRegressor(
    regressor=model, transformer=sklearn.preprocessing.StandardScaler()
  )
  scores = sklearn.model_selection.cross_val_score(pipeline, X, Y, cv=5, scoring="r2")
  return sum(scores) / len(scores)


score(sklearn.linear_model.LinearRegression())

### L1 and L2 Regularization

Now train models with L1 regularization, L2 regularization, then both together.

In [ ]:
# your code here

#### Solution

In [ ]:
score_l1 = score(sklearn.linear_model.Lasso())
score_l2 = score(sklearn.linear_model.Ridge())
score_l1_l2 = score(sklearn.linear_model.ElasticNet())
print("Scores l1, l2 et l1+l2 :", score_l1, score_l2, score_l1_l2)

### Hyperparameter Search

`scikit-learn` contains a few hyperparameter search algorithms, but they're fairly limited (random search & grid search). Several third-party libraries are available to provide this feature.

- *Implement a function that trains a model given hyperparameters and returns a score that will be minimal for your best model*
- *To go further: as in one of the previous exercises, use a pipeline to apply the preprocessing in isolation on each fold of the cross-validation.*
- *To go even further: [Nevergrad](https://facebookresearch.github.io/nevergrad/), a library from Facebook AI Research. Use Nevergrad to find the optimal parameters given a loss function you'll define as the opposite of the score function.*

In [ ]:
# your code here

#### Solution

In [ ]:
# Simple model
model = sklearn.linear_model.Ridge()
params = dict(alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3])
rscv = sklearn.model_selection.RandomizedSearchCV(
  model, params, scoring="r2", n_iter=6, verbose=0
)
search = rscv.fit(X_train, Y_train_scaled)
print(f"Parameters found by RSCV: {search.best_params_}")
print(f"score: {search.best_score_}")
print("=" * 80)

# Complex model that uses a transformation of the targets
model = sklearn.compose.TransformedTargetRegressor(
  regressor=sklearn.linear_model.Ridge(),
  transformer=sklearn.preprocessing.StandardScaler(),
)
params = dict(regressor__alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3])
rscv = sklearn.model_selection.RandomizedSearchCV(
  model, params, scoring="r2", n_iter=6, verbose=0
)
search = rscv.fit(X_train, Y_train)
print(f"Parameters found by RSCV: {search.best_params_}")
print(f"score: {search.best_score_}")
print("=" * 80)


# With Nevergrad
def loss(alpha: float) -> float:
  return -score(sklearn.linear_model.Ridge(alpha=alpha))


parametrization = nevergrad.p.Instrumentation(
  alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)
best_model.fit(X_train, Y_train_scaled)
print(f"Parameters found by Nevergrad: {recommendation.kwargs}")
print(f"score: {score(best_model)}")

### Feature Importance

Use scikit-learn's permutation importance measure ([`sklearn.inspection.permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)) to assess which features are most useful to the best trained model after the hyper-parameter search.

In [ ]:
# your code here

#### Solution

In [ ]:
feature_importances = sklearn.inspection.permutation_importance(
  best_model, X_train, Y_train_scaled, n_repeats=50
)

In [ ]:
series = pd.Series(feature_importances.importances_mean, index=columns)
series = series.sort_values(ascending=False).iloc[:10]
series.plot.bar()

### Polynomial features

In addition to the manual feature extension work, it is possible to use scikit-learn to extend the learning matrix with polynomial features and its class [`sklearn.preprocessing.PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

- *Extend the `X_train` and `X_test` matrices restricted to the top 10 features with polynomial features to create `X_train_poly` and `X_test_poly`*.
- *Train an L1 and L2 regularised model on these data*.

In [ ]:
# your code here

#### Solution

Selection of the 10 most important columns according to row permutation :

In [ ]:
top10 = (-feature_importances.importances_mean).argsort()[:10]
X_train_top10 = X_train[:, top10]
X_test_top10 = X_test[:, top10]

Polynomial features creation :

In [ ]:
def poly_features(array: np.ndarray) -> np.ndarray:
  polynomial_features = sklearn.preprocessing.PolynomialFeatures(
    2, interaction_only=True
  )
  return polynomial_features.fit_transform(array)


X_train_poly = poly_features(X_train_top10)
X_test_poly = poly_features(X_test_top10)

With RandomSearchCV :

In [ ]:
ridge = sklearn.linear_model.Ridge()

# Parameters to adjust
distributions = dict(alpha=scipy.stats.uniform(loc=0, scale=50))

# Optimization procedure call
clf = sklearn.model_selection.RandomizedSearchCV(ridge, distributions)
search = clf.fit(X_train_poly, Y_train_scaled)

# Results
search.best_score_

With Nevergrad.

In [ ]:
parametrization = nevergrad.p.Instrumentation(
  alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)

print(f"Parameters found by Nevergrad : {recommendation.kwargs}")
print(f"score: {score(best_model, X=X_train_poly)}")